# Segmentation Mask Figure
Produces a figure showing original street-view images alongside their panoptic segmentation masks,
stratified by safety class (High / Medium / Low), for both Human and GPT-5.2 rankings.
Layout mirrors Figure 2 from supervisor feedback: raw image row on top, mask row below.

In [ ]:
# ── Cell 1: Imports and configuration ────────────────────────────────────────
import io
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
from azure.storage.blob import BlobServiceClient

# ── Azure ─────────────────────────────────────────────────────────────────────
AZURE_CONN_STR = (
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
BLOB_CONTAINER    = 'london'
IMAGE_PREFIX      = 'training/training/Images/'
PANOPTIC_JSON_DIR = 'PanopticSegmentation/json/'
PANOPTIC_MASK_DIR = 'PanopticSegmentation/'

# ── Local paths ───────────────────────────────────────────────────────────────
GPT_RANK_PATH   = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\gpt_trueskill_rankings_final_20_06_26.csv'
HUMAN_RANK_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\human_trueskill_rankings_final_20_06_26.csv'
OUTPUT_DIR      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Hardcoded point IDs from Figure 6 ────────────────────────────────────────
GPT_POINT_IDS = {
    'High safety'  : ['3682', '1688'],
    'Medium safety': ['516',  '3921'],
    'Low safety'   : ['4561', '1593'],
}
HUMAN_POINT_IDS = {
    'High safety'  : ['1930', '519'],
    'Medium safety': ['1334', '1117'],
    'Low safety'   : ['563',  '2166'],
}

# ── Safety class colours ──────────────────────────────────────────────────────
SAFETY_COLORS = {
    'High safety'  : '#1a7837',
    'Medium safety': '#e08214',
    'Low safety'   : '#b2182b',
}

# ── Figure settings ───────────────────────────────────────────────────────────
DPI         = 300
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

print('Configuration loaded.')

In [ ]:
# ── Cell 2: Azure connection and fetch functions ──────────────────────────────
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)


def fetch_image(image_name: str) -> Image.Image:
    """Fetch original street-view image from Azure."""
    data = container_client.download_blob(
        IMAGE_PREFIX + image_name, timeout=30
    ).readall()
    return Image.open(io.BytesIO(data)).convert('RGB')


def fetch_bytes(blob_path: str) -> bytes:
    """Generic blob fetch — returns raw bytes."""
    return container_client.download_blob(blob_path, timeout=30).readall()


def blob_exists(blob_path: str) -> bool:
    """Check whether a blob path exists in the container."""
    try:
        container_client.get_blob_client(blob_path).get_blob_properties()
        return True
    except Exception:
        return False


print('Azure connection established.')
print(f'Container: {BLOB_CONTAINER}')

In [ ]:
# ── Cell 3: Load ranking CSVs and find best heading per point ID ──────────────
gpt_df   = pd.read_csv(GPT_RANK_PATH)
human_df = pd.read_csv(HUMAN_RANK_PATH)

# Normalise image_name to basename only
for df in [gpt_df, human_df]:
    df['image_name'] = df['image_name'].apply(
        lambda x: os.path.basename(str(x).strip())
    )
    df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')[0]

print('GPT ranking columns  :', list(gpt_df.columns))
print('Human ranking columns:', list(human_df.columns))
print(f'GPT rows   : {len(gpt_df):,}')
print(f'Human rows : {len(human_df):,}')
print(gpt_df.head(3))

In [ ]:
# ── Cell 4: Identify sigma column names and find best heading per point ───────
# After running Cell 3 you will see the column names printed above.
# Set these to match what you see:
GPT_SIGMA_COL   = 'gpt_sigma'    # change if different
HUMAN_SIGMA_COL = 'human_sigma'  # change if different


def best_heading(df: pd.DataFrame, point_id: str, sigma_col: str) -> str:
    """Return the image_name with lowest sigma for a given point_id."""
    grp = df[df['point_id'] == point_id]
    if len(grp) == 0:
        raise ValueError(f'Point ID {point_id} not found in ranking dataframe.')
    return grp.loc[grp[sigma_col].idxmin(), 'image_name']


# Build lookup: {source: {category: [image_name, image_name]}}
image_lookup = {'GPT': {}, 'Human': {}}

for cat, pids in GPT_POINT_IDS.items():
    image_lookup['GPT'][cat] = [
        best_heading(gpt_df, pid, GPT_SIGMA_COL) for pid in pids
    ]

for cat, pids in HUMAN_POINT_IDS.items():
    image_lookup['Human'][cat] = [
        best_heading(human_df, pid, HUMAN_SIGMA_COL) for pid in pids
    ]

print('Image lookup (best heading per point):')
for source, cats in image_lookup.items():
    print(f'  {source}:')
    for cat, imgs in cats.items():
        print(f'    {cat}: {imgs}')

In [ ]:
# ── Cell 5: DIAGNOSTIC — inspect JSON structure for one image ─────────────────
# Uses the first GPT high-safety image to discover the mask filename convention.
# Read the output carefully before running Cell 6.

test_img   = image_lookup['GPT']['High safety'][0]          # e.g. '3682_90.jpg'
test_stem  = test_img.replace('.jpg', '')                   # e.g. '3682_90'
json_path  = PANOPTIC_JSON_DIR + test_stem + '.json'        # e.g. 'PanopticSegmentation/json/3682_90.json'

print(f'Test image : {test_img}')
print(f'JSON path  : {json_path}')
print(f'JSON exists: {blob_exists(json_path)}')
print()

if blob_exists(json_path):
    raw_json = fetch_bytes(json_path).decode('utf-8')
    data     = json.loads(raw_json)
    print('JSON top-level keys:', list(data.keys()))
    print()
    # Print full JSON (truncated at 3000 chars for readability)
    print(json.dumps(data, indent=2)[:3000])
else:
    # Try alternative JSON path without subdirectory
    alt_json = PANOPTIC_MASK_DIR + test_stem + '.json'
    print(f'Trying alternative: {alt_json}')
    print(f'Exists: {blob_exists(alt_json)}')
    if blob_exists(alt_json):
        raw_json = fetch_bytes(alt_json).decode('utf-8')
        data     = json.loads(raw_json)
        print('JSON top-level keys:', list(data.keys()))
        print(json.dumps(data, indent=2)[:3000])

In [ ]:
# ── Cell 6: Discover mask blob path from diagnostic output ────────────────────
# After running Cell 5, look at the JSON output.
# Find the field that contains the mask filename (often 'file_name' or 'mask_path').
# Then check several candidate paths to find where the mask PNG actually lives.

test_img  = image_lookup['GPT']['High safety'][0]
test_stem = test_img.replace('.jpg', '')

# Reload JSON
json_path = PANOPTIC_JSON_DIR + test_stem + '.json'
if not blob_exists(json_path):
    json_path = PANOPTIC_MASK_DIR + test_stem + '.json'

raw_json = fetch_bytes(json_path).decode('utf-8')
data     = json.loads(raw_json)

# Extract mask filename from JSON if present
mask_filename_in_json = data.get('file_name') or data.get('mask_path') or data.get('mask') or 'NOT FOUND'
print(f'Mask field in JSON: {mask_filename_in_json}')
print()

# Try all plausible mask paths
candidates = [
    PANOPTIC_MASK_DIR + test_stem + '.png',
    PANOPTIC_MASK_DIR + test_stem + '.jpg',
    PANOPTIC_MASK_DIR + test_img,
    PANOPTIC_MASK_DIR + 'masks/' + test_stem + '.png',
    PANOPTIC_MASK_DIR + 'Masks/' + test_stem + '.png',
    'PanopticSegmentation/masks/' + test_stem + '.png',
]
if mask_filename_in_json != 'NOT FOUND':
    candidates.insert(0, PANOPTIC_MASK_DIR + mask_filename_in_json)
    candidates.insert(0, mask_filename_in_json)

print('Checking candidate mask paths:')
found_mask_path = None
for c in candidates:
    exists = blob_exists(c)
    print(f'  {exists}  {c}')
    if exists and found_mask_path is None:
        found_mask_path = c

print()
if found_mask_path:
    print(f'CONFIRMED mask path pattern: {found_mask_path}')
    # Derive the template by replacing the stem
    MASK_PATH_TEMPLATE = found_mask_path.replace(test_stem, '{stem}')
    print(f'MASK_PATH_TEMPLATE = "{MASK_PATH_TEMPLATE}"')
    print('Copy this value into Cell 7.')
else:
    print('NO MASK FOUND. List blobs under PanopticSegmentation/ to investigate:')
    blobs = list(container_client.list_blobs(name_starts_with='PanopticSegmentation/'))
    for b in blobs[:30]:
        print(f'  {b.name}')
    print(f'  ... ({len(blobs)} total blobs)')

In [ ]:
# ── Cell 7: Set mask path template and define colour map ─────────────────────
# Paste the MASK_PATH_TEMPLATE value confirmed in Cell 6 here:
MASK_PATH_TEMPLATE = 'PanopticSegmentation/{stem}.png'   # <-- update if needed


def fetch_mask_as_rgb(image_name: str, data: dict) -> Image.Image:
    """
    Fetch the panoptic mask PNG and convert pixel IDs to RGB colours.
    The mask PNG contains integer segment IDs per pixel.
    The JSON data maps segment IDs to class names.
    Returns an RGB PIL Image.
    """
    stem      = image_name.replace('.jpg', '')
    mask_path = MASK_PATH_TEMPLATE.format(stem=stem)
    mask_data = fetch_bytes(mask_path)
    mask_img  = Image.open(io.BytesIO(mask_data))
    mask_arr  = np.array(mask_img)                  # shape (H, W) or (H, W, 3)

    # If mask is RGB-encoded (each segment ID encoded in R+G*256+B*256^2)
    if mask_arr.ndim == 3:
        seg_ids = (mask_arr[:, :, 0].astype(np.int32)
                   + mask_arr[:, :, 1].astype(np.int32) * 256
                   + mask_arr[:, :, 2].astype(np.int32) * 256 * 256)
    else:
        seg_ids = mask_arr.astype(np.int32)

    # Build id -> colour mapping from JSON
    id_to_color = {}
    for seg in data.get('segments_info', []):
        label = seg['label_name']
        color = PANOPTIC_COLORS.get(label, [128, 128, 128])
        id_to_color[seg['id']] = color

    # Render RGB image
    h, w   = seg_ids.shape
    rgb    = np.zeros((h, w, 3), dtype=np.uint8)
    for seg_id, color in id_to_color.items():
        mask = seg_ids == seg_id
        rgb[mask] = color

    return Image.fromarray(rgb)


def fetch_json_for_image(image_name: str) -> dict:
    """Fetch and parse the panoptic JSON for a given image filename."""
    stem      = image_name.replace('.jpg', '')
    json_path = PANOPTIC_JSON_DIR + stem + '.json'
    if not blob_exists(json_path):
        json_path = PANOPTIC_MASK_DIR + stem + '.json'
    raw = fetch_bytes(json_path).decode('utf-8')
    return json.loads(raw)


# ── Panoptic colour map (33 classes) ─────────────────────────────────────────
# Colours chosen to match CityScapes-style palette used in supervisor Figure 2
PANOPTIC_COLORS = {
    'Road'                  : [128, 64,  128],
    'Sidewalk'              : [244, 35,  232],
    'Building'              : [70,  70,  70 ],
    'Wall'                  : [102, 102, 156],
    'Fence'                 : [190, 153, 153],
    'Pole'                  : [153, 153, 153],
    'Utility Pole'          : [120, 120, 120],
    'Traffic Sign (Front)'  : [220, 220,  0 ],
    'Traffic Sign (Back)'   : [180, 180,  0 ],
    'Vegetation'            : [107, 142,  35],
    'Terrain'               : [152, 251, 152],
    'Sky'                   : [70,  130, 180],
    'Person'                : [220,  20,  60],
    'Bicyclist'             : [255,   0,   0],
    'Car'                   : [  0,   0, 142],
    'Truck'                 : [  0,   0,  70],
    'Bus'                   : [  0,  60, 100],
    'Motorcycle'            : [  0,   0, 230],
    'Bicycle'               : [119,  11,  32],
    'Bike Lane'             : [  0, 200, 200],
    'Lane Marking - General': [255, 255, 255],
    'Parking'               : [250, 170, 160],
    'Pedestrian Area'       : [230, 150, 140],
    'Street Light'          : [255, 228, 196],
    'CCTV Camera'           : [200, 200, 200],
    'Billboard'             : [255, 140,   0],
    'Trash Can'             : [160,  82,  45],
    'Catch Basin'           : [100, 100, 100],
    'Curb'                  : [196, 196, 196],
    'Curb Cut'              : [210, 210, 210],
    'Manhole'               : [ 90,  90,  90],
    'Ego Vehicle'           : [ 32,  32,  32],
    'Ground Animal'         : [220, 120,  60],
    'Car Mount'             : [ 50,  50,  50],
}

print(f'Colour map defined for {len(PANOPTIC_COLORS)} classes.')
print(f'Mask path template: {MASK_PATH_TEMPLATE}')

In [ ]:
# ── Cell 8: Quick mask render test ───────────────────────────────────────────
# Test that mask fetch and colour rendering works for one image before
# building the full figure.

test_img  = image_lookup['GPT']['High safety'][0]
print(f'Testing mask render for: {test_img}')

test_json = fetch_json_for_image(test_img)
test_orig = fetch_image(test_img)
test_mask = fetch_mask_as_rgb(test_img, test_json)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor('white')
axes[0].imshow(test_orig)
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')
axes[1].imshow(test_mask)
axes[1].set_title('Panoptic mask', fontsize=10)
axes[1].axis('off')
plt.tight_layout()
plt.show()
print('Test passed.')

In [ ]:
# ── Cell 9: Build full figure ─────────────────────────────────────────────────
# Layout (matching supervisor Figure 2):
#   Two panels: Panel A (Human) and Panel B (GPT-5.2)
#   Each panel: 3 safety classes x 2 locations = 6 columns
#               2 rows per panel: original image (top) + mask (bottom)
#   Dashed vertical separators between safety classes
#   Shared legend at bottom
#   Class labels above each pair of columns

SAFETY_CLASSES = ['High safety', 'Medium safety', 'Low safety']
N_COLS         = 6   # 2 images per safety class x 3 classes
N_IMG_ROWS     = 2   # original + mask
N_PANELS       = 2   # Human + GPT

# Pre-fetch all images and masks
print('Fetching images and masks from Azure...')
fetched = {}   # key: image_name, value: {'orig': PIL, 'mask': PIL, 'json': dict}

all_images = []
for cat in SAFETY_CLASSES:
    all_images += image_lookup['GPT'][cat]
    all_images += image_lookup['Human'][cat]
all_images = list(set(all_images))

for img_name in all_images:
    print(f'  {img_name}', end=' ')
    try:
        j    = fetch_json_for_image(img_name)
        orig = fetch_image(img_name)
        mask = fetch_mask_as_rgb(img_name, j)
        fetched[img_name] = {'orig': orig, 'mask': mask, 'json': j}
        print('OK')
    except Exception as e:
        fetched[img_name] = {'orig': None, 'mask': None, 'json': {}}
        print(f'FAILED: {e}')

print(f'\nFetched {sum(1 for v in fetched.values() if v["orig"] is not None)} / {len(all_images)} images.')

In [ ]:
# ── Cell 10: Render and save figure ──────────────────────────────────────────

def draw_panel(fig, gs, source_label, point_ids_dict, lookup):
    """
    Draw one panel (Human or GPT) into the given GridSpec.
    gs has shape (2, 6): row 0 = originals, row 1 = masks.
    Columns 0-1 = High, 2-3 = Medium, 4-5 = Low.
    """
    col = 0
    for cat_idx, cat in enumerate(SAFETY_CLASSES):
        color    = SAFETY_COLORS[cat]
        img_list = lookup[cat]  # list of 2 image names

        for img_idx, img_name in enumerate(img_list):
            entry = fetched.get(img_name, {})
            orig  = entry.get('orig')
            mask  = entry.get('mask')

            # ── Original image (top row) ──────────────────────────────────
            ax_orig = fig.add_subplot(gs[0, col])
            ax_orig.set_facecolor('white')
            if orig is not None:
                ax_orig.imshow(orig)
            else:
                ax_orig.text(0.5, 0.5, 'Load failed',
                             ha='center', va='center',
                             transform=ax_orig.transAxes,
                             fontsize=7, color='red')
            for spine in ax_orig.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)
            ax_orig.set_xticks([])
            ax_orig.set_yticks([])

            # Class label above first image in each class group
            if img_idx == 0:
                ax_orig.set_title(
                    cat, fontsize=8.5, fontweight='bold',
                    color=color, pad=4
                )

            # ── Mask (bottom row) ─────────────────────────────────────────
            ax_mask = fig.add_subplot(gs[1, col])
            ax_mask.set_facecolor('white')
            if mask is not None:
                ax_mask.imshow(mask)
            else:
                ax_mask.text(0.5, 0.5, 'Mask failed',
                             ha='center', va='center',
                             transform=ax_mask.transAxes,
                             fontsize=7, color='red')
            for spine in ax_mask.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(2.5)
            ax_mask.set_xticks([])
            ax_mask.set_yticks([])

            col += 1

    # Panel label on the left side
    ax_lbl = fig.add_subplot(gs[:, :])
    ax_lbl.set_axis_off()
    ax_lbl.set_title(
        source_label,
        fontsize=10, fontweight='bold', pad=12, loc='left'
    )


# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(
    3, 1, figure=fig,
    height_ratios=[1, 1, 0.06],
    hspace=0.18
)

gs_human = gridspec.GridSpecFromSubplotSpec(
    2, 6, subplot_spec=outer[0],
    hspace=0.04, wspace=0.04
)
gs_gpt = gridspec.GridSpecFromSubplotSpec(
    2, 6, subplot_spec=outer[1],
    hspace=0.04, wspace=0.04
)

draw_panel(fig, gs_human,
           'Panel A — Human crowdsourced votes',
           HUMAN_POINT_IDS, image_lookup['Human'])

draw_panel(fig, gs_gpt,
           'Panel B — GPT-5.2 synthetic votes',
           GPT_POINT_IDS, image_lookup['GPT'])

# ── Dashed vertical separators between safety classes ─────────────────────────
for panel_gs in [gs_human, gs_gpt]:
    for div_col in [2, 4]:   # after column 2 and after column 4
        # Get subplot spec position to draw separator
        sp  = panel_gs[0, div_col].get_position(fig)
        x   = sp.x0 - 0.005
        y0  = panel_gs[1, div_col].get_position(fig).y0
        y1  = panel_gs[0, div_col].get_position(fig).y1
        fig.add_artist(
            plt.Line2D(
                [x, x], [y0, y1],
                transform=fig.transFigure,
                color='#aaaaaa', linewidth=1.2, linestyle='--'
            )
        )

# ── Shared legend ─────────────────────────────────────────────────────────────
ax_legend = fig.add_subplot(outer[2])
ax_legend.set_axis_off()

# Build legend from panoptic classes (show most visually important ones)
legend_classes = [
    'Road', 'Sidewalk', 'Building', 'Vegetation', 'Sky',
    'Car', 'Person', 'Bicycle', 'Bike Lane', 'Terrain',
    'Wall', 'Fence', 'Pole', 'Traffic Sign (Front)', 'Bus',
]
legend_handles = [
    mpatches.Patch(
        color=[c / 255 for c in PANOPTIC_COLORS.get(cls, [128, 128, 128])],
        label=cls
    )
    for cls in legend_classes
]
ax_legend.legend(
    handles=legend_handles,
    loc='center', ncol=8,
    fontsize=7.5, frameon=True,
    framealpha=0.95, edgecolor='#cccccc',
    columnspacing=0.8, handlelength=1.2
)

# ── Save ──────────────────────────────────────────────────────────────────────
out_png = os.path.join(OUTPUT_DIR, 'figure_segmentation_masks.png')
out_pdf = os.path.join(OUTPUT_DIR, 'figure_segmentation_masks.pdf')

fig.savefig(out_png, dpi=DPI, bbox_inches='tight', facecolor='white')
fig.savefig(out_pdf, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved PNG: {out_png}')
print(f'Saved PDF: {out_pdf}')

## Troubleshooting

**If Cell 5 shows JSON not found:** The JSON folder path may differ. Cell 6 will list the first 30 blobs under `PanopticSegmentation/` so you can see the actual structure.

**If Cell 6 shows no mask found:** Run this in a new cell to list all blobs and find the mask:
```python
blobs = list(container_client.list_blobs(name_starts_with='PanopticSegmentation/'))
for b in blobs[:50]:
    print(b.name)
```
Then update `MASK_PATH_TEMPLATE` in Cell 7 accordingly.

**If masks render as grey only:** The segment IDs in the JSON may not match the pixel values in the mask. Print `data['segments_info'][:5]` and `np.unique(mask_arr)[:10]` to compare.

**If sigma column names differ:** Cell 3 prints column names. Update `GPT_SIGMA_COL` and `HUMAN_SIGMA_COL` in Cell 4 to match.

**Suggested caption for the paper:**

Figure X. Street-level images and panoptic segmentation masks for representative locations across each safety class. Panel A shows locations from the human crowdsourced TrueSkill ranking and Panel B shows locations from the GPT-5.2 synthetic TrueSkill ranking. Each column shows one location: the top row displays the original Google Street View image and the bottom row displays the corresponding panoptic segmentation mask produced by the feature extraction pipeline. Columns are grouped by safety class (high safety, medium safety, low safety) with dashed separators. The same representative locations are used as in Figure 6. Segmentation classes follow the 33-class schema applied across the full dataset.